## Cell 1: Environment and Dataset Generation
As per Step 1, we create a synthetic dataset where the relationship is non-linear (output increases early and slows later). We use a square root or a similar curve to ensure a flat linear model would fail.

In [1]:
import numpy as np

# Step 1.1: Input generation
# Create 50 samples between 0 and 10
X = np.linspace(0, 10, 50).reshape(-1, 1)

# Step 1.2: Target generation (Non-linear)
# y = sqrt(x) + some noise
y = np.sqrt(X) + np.random.normal(0, 0.1, X.shape)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (50, 1)
y shape: (50, 1)


## Cell 2: Activation Functions and Derivatives
To "bend" the model, we need non-linear activations. We implement ReLU (Rectified Linear Unit) and its derivative (slope). The slope is essential for the backward pass to tell the model how to adjust weights.

In [2]:
# Step 3 and 4: Activation and Slopes
def relu(z):
    # f(z) = max(0, z)
    return np.maximum(0, z)

def relu_slope(z):
    # derivative of relu: 1 if z > 0 else 0
    return (z > 0).astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_slope(z):
    s = sigmoid(z)
    return s * (1 - s)

## Cell 3: Parameter Initialization
We initialize weights for a network with one hidden layer. Weights are initialized with small random values to "break symmetry," and biases are initialized to zero.

In [3]:
# Step 6: Initialize Parameters
input_dim = 1
hidden_dim = 4
output_dim = 1

# Layer 1 weights and bias
W1 = np.random.randn(input_dim, hidden_dim) * 0.1
b1 = np.zeros((1, hidden_dim))

# Layer 2 weights and bias
W2 = np.random.randn(hidden_dim, output_dim) * 0.1
b2 = np.zeros((1, output_dim))

print("W1 shape:", W1.shape)
print("W2 shape:", W2.shape)

W1 shape: (1, 4)
W2 shape: (4, 1)


## Cell 4: The Forward Pass
Step 7 implementation. The data flows through the first linear transformation, hits the "bending" activation function, and then goes through the final linear output layer.

In [4]:
# Step 7: Forward Pass Function
def forward_pass(X, W1, b1, W2, b2):
    # Input to Hidden
    z1 = X @ W1 + b1
    h = relu(z1)

    # Hidden to Output
    y_hat = h @ W2 + b2
    return z1, h, y_hat

# Test forward pass
z1, h, y_hat = forward_pass(X, W1, b1, W2, b2)
print("Initial prediction shape:", y_hat.shape)

Initial prediction shape: (50, 1)


## Cell 5: The Backward Pass (Gradients)
Step 9 implementation using the Chain Rule. We calculate the error at the output and propagate it backward to find how much $W2, b2, W1,$ and $b1$ contributed to the mistake.

In [5]:
# Step 9: Backward Pass (Manual Derivatives)
def backward_pass(X, y, y_hat, h, z1, W2):
    N = X.shape[0]

    # 1. Error at output
    error = y_hat - y
    dL_dyhat = 2 * error / N

    # 2. Gradients for Layer 2
    dL_dW2 = h.T @ dL_dyhat
    dL_db2 = np.sum(dL_dyhat, axis=0, keepdims=True)

    # 3. Propagate error to hidden layer
    dL_dh = dL_dyhat @ W2.T

    # 4. Backprop through ReLU non-linearity
    dL_dz1 = dL_dh * relu_slope(z1)

    # 5. Gradients for Layer 1
    dL_dW1 = X.T @ dL_dz1
    dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)

    return dL_dW1, dL_db1, dL_dW2, dL_db2

## Cell 6: Training Loop
Step 10 & 11. We combine everything into a loop. In each epoch, we move the parameters in the opposite direction of the gradient ($\theta = \theta - \eta \cdot \nabla$).

In [6]:
# Step 10 and 11: Training Loop
learning_rate = 0.01
epochs = 1000

for epoch in range(epochs):
    # Forward pass
    z1, h, y_hat = forward_pass(X, W1, b1, W2, b2)

    # Compute Loss (MSE)
    loss = np.mean((y_hat - y)**2)

    # Backward pass
    dW1, db1, dW2, db2 = backward_pass(X, y, y_hat, h, z1, W2)

    # Parameter Update
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 0, Loss: 4.9232
Epoch 100, Loss: 0.0576
Epoch 200, Loss: 0.0445
Epoch 300, Loss: 0.0407
Epoch 400, Loss: 0.0396
Epoch 500, Loss: 0.0393
Epoch 600, Loss: 0.0392
Epoch 700, Loss: 0.0392
Epoch 800, Loss: 0.0392
Epoch 900, Loss: 0.0392


## Cell 7: Final Reflection and Policy
After training, we observe if the model's output line is now curved (bent) to match the data.

In [7]:
# Final Checkpoint
_, _, final_predictions = forward_pass(X, W1, b1, W2, b2)

print("Final Predictions (first 5):")
print(final_predictions[:5])

# Comment on "Bending":
# Because we used ReLU and a hidden layer, the model is no longer a straight line.
# It has learned a piecewise linear representation that 'bends' at the neurons' activation points.

Final Predictions (first 5):
[[0.82663963]
 [0.87862694]
 [0.93061424]
 [0.98260155]
 [1.03458886]]
